# M3 step 2: HiDDeN with a U-Net encoder

This notebook replaces the convolutional encoder from step 1 with a U-Net-style encoder. It uses the same data splits, decoder, discriminator, losses, batch size and 60-epoch training budget.

The U-Net gave better clean-image message recovery in this comparison. It is also much larger than the baseline encoder, so the results compare these two configurations rather than isolating network shape from model capacity. Step 3 uses its trained weights for JPEG and resize fine-tuning.

## Why use a U-Net encoder?

The convolutional baseline keeps the image at 128×128 throughout its feature layers. This encoder instead builds features at 128×128, 64×64 and 32×32, then upsamples them back to the original size. Skip connections join the earlier features to the upsampled features.

This gives the encoder a wider spatial context while retaining finer detail. The message is supplied at the input, at the smallest feature map and before the final output layers. The design is inspired by StegaStamp, but it is not a copy of that model.

## Training setup and time

The run used PyTorch 2.4.1 on an NVIDIA GeForce RTX 2070 Super with Max-Q Design. The 60 logged epochs took about 4 hours 40 minutes. The encoder has 2,625,795 parameters, compared with 170,499 in the convolutional baseline.

## 1. Settings

The training settings match step 1: 32 message bits, batch size 32, Adam with a learning rate of 0.001 and 60 epochs. Only the encoder choice changes.

In [ ]:
import os, io, csv, json, time, math, random, shutil, hashlib, zipfile, urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

ENCODER = 'unet'
NOISE = 'none'            # clean photos, no distortion
INIT_FROM = None          # start from random weights
RUN_NAME = f'hidden_{ENCODER}_{NOISE}_10k'

SEED = 42
N_TRAIN = 10000
IMAGE_SIZE = 128
MESSAGE_BITS = 32
BATCH_SIZE = 32
EPOCHS = 60
LEARNING_RATE = 1e-3
ENCODER_LOSS, DECODER_LOSS, ADVERSARIAL_LOSS = 0.7, 1.0, 1e-3
NUM_WORKERS = 0
RESUME = True

COCO_ROOT = Path(r'E:\VeriStamp_M3\coco')
MANIFEST = Path(r'E:\VeriStamp_M3\VeriStamp_M3_runs\pilot_clean_4k_v2\split_manifest.csv')
DATA = Path(r'E:\VeriStamp_M3\hidden_data')
RUN_DIR = Path(r'E:\VeriStamp_M3\hidden_runs') / RUN_NAME

assert ENCODER in ('conv', 'unet') and NOISE in ('none', 'jpeg')
assert (COCO_ROOT / 'val2017').exists(), f'val2017 not found in {COCO_ROOT}'
assert MANIFEST.exists(), f'Split manifest not found: {MANIFEST}'
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Switch the kernel to "Deep Learning".')
device = torch.device('cuda')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG = RUN_DIR / 'train_log.txt'
def log(message):
    print(message, flush=True)
    with open(LOG, 'a', encoding='utf-8') as f:
        f.write(message + '\n')

config = {'encoder': ENCODER, 'noise': NOISE, 'init_from': str(INIT_FROM) if INIT_FROM else None,
          'seed': SEED, 'n_train': N_TRAIN, 'image_size': IMAGE_SIZE, 'message_bits': MESSAGE_BITS,
          'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
          'encoder_loss': ENCODER_LOSS, 'decoder_loss': DECODER_LOSS, 'adversarial_loss': ADVERSARIAL_LOSS}
print('PyTorch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
print(f'Free space on E: {shutil.disk_usage(DATA.anchor).free / 1024**3:.1f} GB (first run needs about 22 GB)')
print('Run folder:', RUN_DIR)

## 2. Data

This run reuses the 10,000 training images and the 500-image validation and test splits prepared for step 1. If the shared folders are complete, this cell checks their counts.

In [ ]:
TRAIN_URL = 'http://images.cocodataset.org/zips/train2017.zip'
folders = {'train': DATA / 'train', 'validation': DATA / 'val', 'test': DATA / 'test'}
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)

def download(url, target):
    if target.exists():
        return
    partial = target.with_suffix('.zip.part')
    request = urllib.request.Request(url, headers={'User-Agent': 'VeriStamp-M3-Research/1.0'})
    print('Downloading', url)
    with urllib.request.urlopen(request, timeout=120) as response, open(partial, 'wb') as out:
        total = int(response.headers.get('Content-Length', 0))
        done, next_report = 0, 0
        while True:
            chunk = response.read(8 * 1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
            done += len(chunk)
            if done >= next_report:
                print(f'  {done / 1024**3:.1f} / {total / 1024**3:.1f} GB', flush=True)
                next_report += 1024**3
    if total and partial.stat().st_size != total:
        raise RuntimeError('Download incomplete. Delete the .part file and rerun this cell.')
    partial.replace(target)

with open(MANIFEST, newline='') as f:
    eval_rows = [r for r in csv.DictReader(f) if r['split'] in ('validation', 'test')]
for row in eval_rows:
    target = folders[row['split']] / row['filename']
    if not target.exists():
        shutil.copy(COCO_ROOT / 'val2017' / row['filename'], target)

if len(list(folders['train'].glob('*.jpg'))) < N_TRAIN:
    archive = COCO_ROOT / 'train2017.zip'
    download(TRAIN_URL, archive)
    with zipfile.ZipFile(archive) as zf:
        members = sorted(m for m in zf.namelist() if m.startswith('train2017/') and m.endswith('.jpg'))
        if len(members) != 118287:
            raise RuntimeError(f'Unexpected train2017 image count: {len(members)}')
        chosen = random.Random(SEED).sample(members, N_TRAIN)
        for i, member in enumerate(chosen, 1):
            target = folders['train'] / Path(member).name
            if not target.exists():
                with zf.open(member) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
            if i % 1000 == 0:
                print(f'  extracted {i}/{N_TRAIN}', flush=True)

counts = {name: len(list(folder.glob('*.jpg'))) for name, folder in folders.items()}
print('Image counts:', counts)
assert counts == {'train': N_TRAIN, 'validation': 500, 'test': 500}, counts

## 3. Loading the photos

As in step 1, training uses random 128×128 crops and new random messages for each batch. Validation uses native-resolution centre crops and fixed random messages. Pixel values are scaled to [−1, 1].

In [ ]:
class Photos(Dataset):
    def __init__(self, folder, train):
        self.files = sorted(Path(folder).glob('*.jpg'))
        crop = transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True) if train else transforms.CenterCrop(IMAGE_SIZE)
        self.transform = transforms.Compose([crop, transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        with Image.open(self.files[index]) as image:
            return self.transform(image.convert('RGB')), index

train_set, val_set = Photos(folders['train'], True), Photos(folders['validation'], False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_messages = torch.randint(0, 2, (len(val_set), MESSAGE_BITS), generator=torch.Generator().manual_seed(SEED)).float()
print('Train', len(train_set), '| validation', len(val_set), '| steps per epoch', len(train_loader))

## 4. Model

### Decoder and discriminator

These networks are unchanged from step 1. The decoder recovers the 32 message bits, and the discriminator distinguishes stamped images from originals.

In [ ]:
class ConvBNRelu(nn.Module):
    def __init__(self, channels_in, channels_out, stride=1):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels_in, channels_out, 3, stride, padding=1),
                                    nn.BatchNorm2d(channels_out), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.layers(x)


class Decoder(nn.Module):
    """Reads the 32 bits back from a stamped photo."""
    def __init__(self, channels=64, blocks=7, message_bits=32):
        super().__init__()
        layers = [ConvBNRelu(3, channels)] + [ConvBNRelu(channels, channels) for _ in range(blocks - 1)]
        layers += [ConvBNRelu(channels, message_bits), nn.AdaptiveAvgPool2d(1)]
        self.layers = nn.Sequential(*layers)
        self.linear = nn.Linear(message_bits, message_bits)

    def forward(self, image):
        return self.linear(self.layers(image).flatten(1))


class Discriminator(nn.Module):
    """Scores whether a photo is stamped or original."""
    def __init__(self, channels=64, blocks=3):
        super().__init__()
        layers = [ConvBNRelu(3, channels)] + [ConvBNRelu(channels, channels) for _ in range(blocks - 1)]
        self.before_linear = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1))
        self.linear = nn.Linear(channels, 1)

    def forward(self, image):
        return self.linear(self.before_linear(image).flatten(1))

### Encoder

The following cell defines the U-Net-style encoder described above. The output is the stamped image itself.

In [ ]:
class UNetEncoder(nn.Module):
    def __init__(self, channels=64, message_bits=32):
        super().__init__()
        c, m = channels, message_bits
        self.inc = nn.Sequential(ConvBNRelu(3 + m, c), ConvBNRelu(c, c))
        self.down1 = nn.Sequential(ConvBNRelu(c, 2 * c, stride=2), ConvBNRelu(2 * c, 2 * c))
        self.down2 = nn.Sequential(ConvBNRelu(2 * c, 4 * c, stride=2), ConvBNRelu(4 * c, 4 * c))
        self.bottleneck = ConvBNRelu(4 * c + m, 4 * c)
        self.up2 = nn.Sequential(ConvBNRelu(4 * c + 2 * c, 2 * c), ConvBNRelu(2 * c, 2 * c))
        self.up1 = nn.Sequential(ConvBNRelu(2 * c + c, c), ConvBNRelu(c, c))
        self.after_concat_layer = ConvBNRelu(c + 3 + m, c)
        self.final_layer = nn.Conv2d(c, 3, kernel_size=1)

    @staticmethod
    def _expand(message, like):
        return message.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, like.shape[-2], like.shape[-1])

    def forward(self, image, message):
        x0 = self.inc(torch.cat([image, self._expand(message, image)], dim=1))       # full size
        x1 = self.down1(x0)                                                          # half size
        x2 = self.down2(x1)                                                          # quarter size
        x2 = self.bottleneck(torch.cat([x2, self._expand(message, x2)], dim=1))
        u2 = self.up2(torch.cat([F.interpolate(x2, size=x1.shape[-2:], mode='nearest'), x1], dim=1))
        u1 = self.up1(torch.cat([F.interpolate(u2, size=x0.shape[-2:], mode='nearest'), x0], dim=1))
        return self.final_layer(self.after_concat_layer(torch.cat([self._expand(message, image), u1, image], dim=1)))

def build_encoder():
    assert ENCODER == 'unet', 'This notebook contains only the U-Net encoder.'
    return UNetEncoder(message_bits=MESSAGE_BITS)

### Simulated JPEG

The JPEG-Mask layer is retained from step 1 but disabled. This run uses no added JPEG or resize distortions.

In [ ]:
def _dct_coeff(n, k, N):
    return np.cos(np.pi / N * (n + 0.5) * k)

def _idct_coeff(n, k, N):
    return (int(n == 0) * (-0.5) + np.cos(np.pi / N * (k + 0.5) * n)) * np.sqrt(1 / (2.0 * N))

def _filters(fn):
    filters = np.zeros((64, 8, 8))
    for ky in range(8):
        for kx in range(8):
            for ny in range(8):
                for nx in range(8):
                    filters[ky * 8 + kx, ny, nx] = fn(ny, ky, 8) * fn(nx, kx, 8)
    return torch.tensor(filters, dtype=torch.float32).unsqueeze(1)

def _zigzag_mask(keep):
    order = sorted(((x, y) for x in range(8) for y in range(8)),
                   key=lambda p: (p[0] + p[1], -p[1] if (p[0] + p[1]) % 2 else p[1]))
    mask = np.zeros((8, 8), dtype=np.float32)
    for i, j in order[:keep]:
        mask[i, j] = 1
    return mask

class JpegCompression(nn.Module):
    def __init__(self, size=128, keep=(25, 9, 9)):
        super().__init__()
        assert size % 8 == 0
        self.register_buffer('dct', _filters(_dct_coeff))
        self.register_buffer('idct', _filters(_idct_coeff))
        self.register_buffer('mask', torch.tensor(np.stack([np.tile(_zigzag_mask(k), (size // 8, size // 8)) for k in keep])))

    @staticmethod
    def _blockwise(image, filters):
        out = []
        for c in range(image.shape[1]):
            x = F.conv2d(image[:, c:c + 1], filters, stride=8)
            b, _, h8, w8 = x.shape
            x = x.permute(0, 2, 3, 1).reshape(b, h8, w8, 8, 8).permute(0, 1, 3, 2, 4).reshape(b, h8 * 8, w8 * 8)
            out.append(x.unsqueeze(1))
        return torch.cat(out, dim=1)

    def forward(self, x):
        r, g, b = x[:, 0], x[:, 1], x[:, 2]
        yuv = torch.stack([0.299 * r + 0.587 * g + 0.114 * b,
                           -0.14713 * r - 0.28886 * g + 0.436 * b,
                           0.615 * r - 0.51499 * g - 0.10001 * b], dim=1)
        y = self._blockwise(self._blockwise(yuv, self.dct) * self.mask, self.idct)
        return torch.stack([y[:, 0] + 1.13983 * y[:, 2],
                            y[:, 0] - 0.39465 * y[:, 1] - 0.58060 * y[:, 2],
                            y[:, 0] + 2.03211 * y[:, 1]], dim=1)

jpeg_layer = JpegCompression(IMAGE_SIZE).to(device) if NOISE == 'jpeg' else None

def apply_noise(encoded):
    if jpeg_layer is not None and random.random() < 0.5:
        return jpeg_layer(encoded)
    return encoded

### Build and check

Create the networks and optimizers, then check the output shapes using two random images and messages.

In [ ]:
encoder, decoder, discriminator = build_encoder().to(device), Decoder(message_bits=MESSAGE_BITS).to(device), Discriminator().to(device)
opt_g = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LEARNING_RATE)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE)
bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()

with torch.no_grad():
    x = torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device) * 2 - 1
    m = torch.randint(0, 2, (2, MESSAGE_BITS), device=device).float()
    stamped = encoder(x, m)
    assert stamped.shape == x.shape and decoder(stamped).shape == m.shape and discriminator(stamped).shape == (2, 1)
    if jpeg_layer is not None:
        assert jpeg_layer(stamped).shape == x.shape
count = lambda net: sum(p.numel() for p in net.parameters())
print(f'Encoder ({ENCODER}) {count(encoder):,} | decoder {count(decoder):,} | discriminator {count(discriminator):,} parameters')

## 5. Training

The training step and losses match step 1:

`loss = 0.7 × image MSE + 1.0 × message MSE + 0.001 × adversarial BCE`

Logs and checkpoints are saved after each epoch. `best.pt` is selected by a strict improvement in clean validation bit accuracy. A later checkpoint with equal accuracy does not replace it, even if PSNR is higher.

In [ ]:
def train_step(images, messages):
    encoder.train(); decoder.train(); discriminator.train()
    ones = torch.ones(images.shape[0], 1, device=device)
    zeros = torch.zeros(images.shape[0], 1, device=device)

    opt_d.zero_grad(set_to_none=True)
    d_cover = bce(discriminator(images), ones)
    d_cover.backward()
    encoded = encoder(images, messages)
    decoded = decoder(apply_noise(encoded))
    d_encoded = bce(discriminator(encoded.detach()), zeros)
    d_encoded.backward()
    opt_d.step()

    opt_g.zero_grad(set_to_none=True)
    g_adv = bce(discriminator(encoded), ones)
    g_enc = mse(encoded, images)
    g_dec = mse(decoded, messages)
    loss = ADVERSARIAL_LOSS * g_adv + ENCODER_LOSS * g_enc + DECODER_LOSS * g_dec
    if not torch.isfinite(loss):
        raise RuntimeError('Loss became NaN/inf. Stop and check this run.')
    loss.backward()
    opt_g.step()
    bits = ((decoded.detach() > 0.5).float() == messages).float().mean().item()
    return {'loss': loss.item(), 'image_mse': g_enc.item(), 'message_mse': g_dec.item(), 'bits': bits,
            'adv': g_adv.item(), 'd_loss': (d_cover.item() + d_encoded.item()) / 2}

def psnr01(original, stamped):
    """PSNR per image, measured on the 0 to 1 pixel scale."""
    err = ((stamped.clamp(-1, 1) - original) / 2).square().flatten(1).mean(1).clamp_min(1e-12)
    return -10 * torch.log10(err)

@torch.no_grad()
def validate():
    encoder.eval(); decoder.eval()
    n, bits, exact, psnr, jpeg_bits = 0, 0.0, 0.0, 0.0, 0.0
    for images, index in val_loader:
        images, messages = images.to(device), val_messages[index].to(device)
        encoded = encoder(images, messages)
        correct = ((decoder(encoded) > 0.5).float() == messages)
        bits += correct.float().mean(1).sum().item()
        exact += correct.all(1).float().sum().item()
        psnr += psnr01(images, encoded).sum().item()
        if jpeg_layer is not None:
            jpeg_bits += ((decoder(jpeg_layer(encoded)) > 0.5).float() == messages).float().mean(1).sum().item()
        n += len(images)
    return {'val_bits': bits / n, 'val_exact': exact / n, 'val_psnr_db': psnr / n,
            'val_jpeg_approx_bits': jpeg_bits / n if jpeg_layer is not None else float('nan')}

def save_checkpoint(path, epoch, history):
    state = {'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(), 'discriminator': discriminator.state_dict(),
             'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(), 'epoch': epoch, 'history': history, 'config': config}
    torch.save(state, str(path) + '.tmp')
    os.replace(str(path) + '.tmp', path)

def write_history(history):
    with open(RUN_DIR / 'history.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2))
print('Training helpers ready.')

In [ ]:
history, first_epoch = [], 1
latest = RUN_DIR / 'latest.pt'
same = lambda a, b: {k: v for k, v in a.items() if k != 'epochs'} == {k: v for k, v in b.items() if k != 'epochs'}
if RESUME and latest.exists():
    state = torch.load(latest, map_location=device)
    if not same(state['config'], config):
        raise RuntimeError('This run folder has different settings. Change RUN_NAME or restore the settings.')
    for net, key in [(encoder, 'encoder'), (decoder, 'decoder'), (discriminator, 'discriminator'), (opt_g, 'opt_g'), (opt_d, 'opt_d')]:
        net.load_state_dict(state[key])
    history, first_epoch = state['history'], state['epoch'] + 1
    log(f'Resuming {RUN_NAME} at epoch {first_epoch}')
elif INIT_FROM:
    state = torch.load(INIT_FROM, map_location=device)
    encoder.load_state_dict(state['encoder']); decoder.load_state_dict(state['decoder']); discriminator.load_state_dict(state['discriminator'])
    log(f'Starting {RUN_NAME} from weights in {INIT_FROM} (epoch {state["epoch"]})')
else:
    log(f'Starting {RUN_NAME} from scratch: {json.dumps(config)}')

best_bits = max([row['val_bits'] for row in history], default=-1.0)
for epoch in range(first_epoch, EPOCHS + 1):
    started = time.time()
    totals, n = {}, 0
    for images, _ in train_loader:
        images = images.to(device, non_blocking=True)
        messages = torch.randint(0, 2, (images.shape[0], MESSAGE_BITS), device=device).float()
        for key, value in train_step(images, messages).items():
            totals[key] = totals.get(key, 0.0) + value * len(images)
        n += len(images)
    row = {'epoch': epoch, **{f'train_{k}': v / n for k, v in totals.items()}, **validate(),
           'minutes': (time.time() - started) / 60}
    history.append(row)
    write_history(history)
    if row['val_bits'] > best_bits:
        best_bits = row['val_bits']
        save_checkpoint(RUN_DIR / 'best.pt', epoch, history)
    save_checkpoint(latest, epoch, history)
    jpeg_text = f" | val JPEG-approx bits {row['val_jpeg_approx_bits']:.2%}" if jpeg_layer is not None else ''
    log(f"Epoch {epoch:3d}/{EPOCHS} | train bits {row['train_bits']:.2%} | val bits {row['val_bits']:.2%} | "
        f"exact {row['val_exact']:.2%} | PSNR {row['val_psnr_db']:.2f} dB{jpeg_text} | {row['minutes']:.2f} min")
log(f'Training finished. Best validation bit accuracy {best_bits:.2%}')

## 6. Results

### Learning curves

Validation bit accuracy rose from 69.34% at epoch 10 to 99.42% at epoch 27. It first reached 100% at epoch 48 and reached 100% again at epochs 50, 58 and 60. Validation PSNR was 32.43 dB at epoch 48 and 33.76 dB at epoch 60.

The checkpoint rule retains epoch 48 for the test below. The fine-tuning run in step 3 starts from the later epoch-60 checkpoint.

In [ ]:
with open(RUN_DIR / 'history.csv', newline='') as f:
    rows = [{k: float(v) for k, v in r.items()} for r in csv.DictReader(f)]
epochs = [r['epoch'] for r in rows]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [r['train_bits'] * 100 for r in rows], label='Train bits')
axes[0].plot(epochs, [r['val_bits'] * 100 for r in rows], label='Validation bits')
if NOISE == 'jpeg':
    axes[0].plot(epochs, [r['val_jpeg_approx_bits'] * 100 for r in rows], label='Validation bits after JPEG approx')
axes[0].set(title='Bit accuracy', xlabel='Epoch', ylabel='%', ylim=(40, 101)); axes[0].legend()
axes[1].plot(epochs, [r['val_psnr_db'] for r in rows], color='#437346')
axes[1].set(title='Validation PSNR (stamped vs original)', xlabel='Epoch', ylabel='dB')
axes[2].plot(epochs, [r['val_exact'] * 100 for r in rows], color='#784c91')
axes[2].set(title='Validation exact 32-bit messages', xlabel='Epoch', ylabel='%')
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(RUN_DIR / 'learning_curves.png', bbox_inches='tight')
plt.show()
last = rows[-5:]
print(f"Last {len(last)} epochs: val bits {np.mean([r['val_bits'] for r in last]):.2%}, "
      f"PSNR {np.mean([r['val_psnr_db'] for r in last]):.2f} dB  (use these to compare with the U-Net notebook)")

### Test with the VeriStamp payload

The payload has 32 bits: 2 for the verdict, 4 for the AI-probability bucket, 4 for the detector ID, 14 for an image tag and 8 for a CRC checksum. These experiments use repeatable sample payloads generated from each filename, rather than predictions from M1, M2 or the fusion model.

The CRC checks for transmission errors, but a matching checksum does not guarantee that every bit is correct. The results therefore report bit accuracy, exact 32-bit recovery and CRC acceptance separately.

The epoch-48 checkpoint is evaluated on the same 500 test images as step 1, with the same resize-and-centre-crop preprocessing. The conditions are clipped float output, a real PNG save and reload, and Pillow JPEG saves at qualities 90 and 70. This is separate from the clean validation test used to select the checkpoint.

In [ ]:
VERDICT_CODES = {'Real': 0, 'AI': 1, 'Uncertain': 2}
CODE_VERDICTS = {v: k for k, v in VERDICT_CODES.items()}

def int_to_bits(value, width):
    return [(value >> s) & 1 for s in range(width - 1, -1, -1)]

def bits_to_int(bits):
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out

def crc8(bits):
    reg = 0
    for b in bits:
        fb = ((reg >> 7) & 1) ^ int(b)
        reg = (reg << 1) & 255
        if fb:
            reg ^= 0x07
    return int_to_bits(reg, 8)

def encode_payload(verdict, p_ai, detector_id=3, image_tag=0):
    fields = (int_to_bits(VERDICT_CODES[verdict], 2) + int_to_bits(round(max(0.0, min(1.0, p_ai)) * 15), 4)
              + int_to_bits(detector_id, 4) + int_to_bits(image_tag & 0x3FFF, 14))
    return fields + crc8(fields)

def decode_payload(bits):
    code = bits_to_int(bits[:2])
    valid = code in CODE_VERDICTS and crc8(bits[:24]) == list(bits[24:])
    return {'valid': valid, 'verdict': CODE_VERDICTS[code] if valid else None}

def fixed_payload(filename):
    digest = hashlib.sha256(f'{SEED}:payload:{filename}'.encode()).digest()
    rng = random.Random(int.from_bytes(digest[:8], 'big'))
    p_ai = rng.random()
    verdict = 'Real' if p_ai < 0.35 else ('AI' if p_ai > 0.65 else 'Uncertain')
    return encode_payload(verdict, p_ai, 3, rng.randrange(2**14))

example = encode_payload('Real', 0.1, image_tag=123)
assert decode_payload(example)['verdict'] == 'Real'
assert all(not decode_payload(example[:i] + [1 - example[i]] + example[i + 1:])['valid'] for i in range(32))
print('Payload functions OK')

In [ ]:
best = torch.load(RUN_DIR / 'best.pt', map_location=device)
encoder.load_state_dict(best['encoder']); decoder.load_state_dict(best['decoder'])
encoder.eval(); decoder.eval()
print('Best checkpoint: epoch', best['epoch'])

test_transform = transforms.Compose([transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor()])
test_files = sorted(folders['test'].glob('*.jpg'))

def save_reload(images01, fmt, quality=None):
    out = []
    for image in images01.cpu():
        array = (image.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
        buffer = io.BytesIO()
        Image.fromarray(array).save(buffer, format=fmt, **({'quality': quality} if fmt == 'JPEG' else {}))
        buffer.seek(0)
        with Image.open(buffer) as restored:
            out.append(torch.from_numpy(np.asarray(restored.convert('RGB'), dtype=np.float32) / 255).permute(2, 0, 1))
    return torch.stack(out).to(images01.device)

@torch.no_grad()
def read_bits(images01):
    return (decoder(images01 * 2 - 1) > 0.5).long().cpu().tolist()

conditions = ['float_clean', 'png_saved', 'jpeg90', 'jpeg70']
records, psnr_sum, unstamped_valid = [], 0.0, 0
for start in range(0, len(test_files), BATCH_SIZE):
    files = test_files[start:start + BATCH_SIZE]
    images01 = []
    for path in files:
        with Image.open(path) as image:
            images01.append(test_transform(image.convert('RGB')))
    images01 = torch.stack(images01).to(device)
    messages = torch.tensor([fixed_payload(p.name) for p in files], dtype=torch.float32, device=device)
    with torch.no_grad():
        stamped01 = ((encoder(images01 * 2 - 1, messages) + 1) / 2).clamp(0, 1)
    png = save_reload(stamped01, 'PNG')
    psnr_sum += (-10 * torch.log10((png - images01).square().flatten(1).mean(1).clamp_min(1e-12))).sum().item()
    unstamped_valid += sum(decode_payload(b)['valid'] for b in read_bits(save_reload(images01, 'PNG')))
    variants = {'float_clean': stamped01, 'png_saved': png,
                'jpeg90': save_reload(stamped01, 'JPEG', 90), 'jpeg70': save_reload(stamped01, 'JPEG', 70)}
    truth = messages.long().cpu().tolist()
    for condition in conditions:
        for path, expected, actual in zip(files, truth, read_bits(variants[condition])):
            got, want = decode_payload(actual), decode_payload(expected)
            records.append({'filename': path.name, 'condition': condition,
                            'bit_accuracy': sum(a == b for a, b in zip(actual, expected)) / 32,
                            'exact_message': int(actual == expected), 'stamp_valid': int(got['valid']),
                            'valid_correct_verdict': int(got['valid'] and got['verdict'] == want['verdict']),
                            'expected_bits': ''.join(map(str, expected)), 'recovered_bits': ''.join(map(str, actual))})

n = len(test_files)
summary = [{'condition': c, 'n_images': n, **{k: float(np.mean([r[k] for r in records if r['condition'] == c]))
            for k in ['bit_accuracy', 'exact_message', 'stamp_valid', 'valid_correct_verdict']}} for c in conditions]
for name, rows_out in [('test_predictions.csv', records), ('test_summary.csv', summary)]:
    with open(RUN_DIR / name, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows_out[0]))
        writer.writeheader()
        writer.writerows(rows_out)
quality = {'n_images': n, 'checkpoint_epoch': best['epoch'], 'png_stamped_psnr_db': psnr_sum / n,
           'unstamped_png_accepted_count': unstamped_valid, 'unstamped_png_acceptance_rate': unstamped_valid / n}
(RUN_DIR / 'quality_and_false_acceptance.json').write_text(json.dumps(quality, indent=2))

lines = [f'{"Condition":<14} {"Bit acc":>10} {"Exact msg":>12} {"CRC-valid":>12} {"Valid+verdict":>15}']
lines += [f'{r["condition"]:<14} {r["bit_accuracy"]:>10.2%} {r["exact_message"]:>12.2%} {r["stamp_valid"]:>12.2%} '
          f'{r["valid_correct_verdict"]:>15.2%}' for r in summary]
lines += [f'Mean stamped PNG PSNR: {quality["png_stamped_psnr_db"]:.2f} dB', f'Unstamped PNGs accepted: {unstamped_valid}/{n}']
log('\n'.join(['Test results (best checkpoint, epoch %d):' % best['epoch']] + lines))

### Example stamped image

The figure shows the original image, the stamped image after PNG saving, and the absolute pixel difference multiplied by 10.

In [ ]:
path = test_files[0]
with Image.open(path) as image:
    original = test_transform(image.convert('RGB')).unsqueeze(0).to(device)
message = torch.tensor([fixed_payload(path.name)], dtype=torch.float32, device=device)
with torch.no_grad():
    stamped = save_reload(((encoder(original * 2 - 1, message) + 1) / 2).clamp(0, 1), 'PNG')
recovered = read_bits(stamped)[0]
print('Expected :', ''.join(map(str, message.long()[0].tolist())))
print('Recovered:', ''.join(map(str, recovered)), '| exact:', recovered == message.long()[0].tolist())
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for axis, img, title in zip(axes, [original[0], stamped[0], ((stamped[0] - original[0]).abs() * 10).clamp(0, 1)],
                            ['Original', 'Stamped (saved as PNG)', 'Difference ×10']):
    axis.imshow(img.permute(1, 2, 0).cpu()); axis.set_title(title); axis.axis('off')
plt.tight_layout()
fig.savefig(RUN_DIR / 'example_comparison.png', bbox_inches='tight')
plt.show()

### Save the results

Package the run folder, including logs, test results, figures and checkpoints, as a ZIP archive.

In [ ]:
bundle = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Saved', bundle, f'({Path(bundle).stat().st_size / 1024**2:.1f} MB)')

## Summary

Results on 500 test images after PNG saving:

| Measurement | Convolutional encoder | U-Net encoder |
|---|---:|---:|
| Bit accuracy | 92.97% | 99.66% |
| Exact 32-bit messages | 13.2% | 94.8% |
| Mean stamped-PNG PSNR | 35.47 dB | 32.65 dB |

The U-Net recovered complete messages more often, with a 2.82 dB reduction in PSNR. We selected it for the next stage on that basis. These are results from one training run per encoder.

Clean-image recovery did not carry over to JPEG compression: bit accuracy was 50.76% at quality 90 and 50.57% at quality 70, with no exact messages in either condition. One of the 500 unstamped images passed the payload check.

The amplified residual shows a band near the image borders. StegaStamp also discusses learned edge patterns, although that does not establish the cause of the pattern in this model. Step 3 evaluates JPEG and resize robustness after fine-tuning.

## Sources

The decoder, discriminator, losses and JPEG-Mask layer use the same [PyTorch HiDDeN implementation](https://github.com/ando-khachatryan/HiDDeN/tree/556f76dd0602e4351ed4c19bd2ee87d50ef3c0de) credited in step 1, where its MIT license is reproduced.

The encoder is inspired by the U-Net design in [StegaStamp: Invisible Hyperlinks in Physical Photographs](https://arxiv.org/abs/1904.05343) by Tancik et al. (CVPR 2020). Our architecture, payload handling and losses differ from StegaStamp.